# Task 3 — AQE Order-Effect Diagnostic: distance_by_weather

## Purpose

This is a diagnostic check, not a benchmark result, and it does not get appended to task3_benchmark_results.jsonl.

The distance_by_weather AQE notebook measured AQE disabled first, then AQE enabled second, and got 1.37x with no corresponding change in the physical plan. Since the plan showed nothing had changed, a genuine 1.37x effect from AQE didn't make sense. Our guess was that whichever configuration runs second benefits from leftover warm-up from the first configuration's own runs.

This notebook tests that directly by reversing the order: AQE enabled first, AQE disabled second.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

from data_intensive_computing.benchmark_helpers import (
    build_spark_session,
    register_delta_tables,
    timed_run,
    capture_explain,
    find_project_root,
)

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

spark = build_spark_session(app_name="Task3-AQE-OrderCheck-DistanceByWeather")
register_delta_tables(spark, PROCESSED_DIR)

:: loading settings :: url = jar:file:/Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rexchang/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rexchang/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fe446534-7e99-438d-ba74-19d153692f93;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in c

Registered taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_trips
Registered weather: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/weather
Registered air_quality: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/air_quality
Registered taxi_zones: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/taxi_zones
Registered integrated_taxi_trips: /Users/rexchang/Documents/KTH/Courses/26-Fall-P1/Data-Intensive-Computing/Homework/data-intensive-computing/data/processed/integrated_taxi_trips


26/09/18 00:15:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Query (identical to the original AQE notebook)

In [2]:
QUERY_SQL = """
WITH classified AS (
    SELECT
        trip_id,
        trip_distance,
        CASE
            WHEN weather_condition_code IN (1, 2) THEN 'clear_or_fair'
            WHEN weather_condition_code IN (3, 4) THEN 'cloudy'
            WHEN weather_condition_code IN (5, 6) THEN 'fog'
            WHEN weather_condition_code IN (7, 8, 9, 10, 11, 17, 18) THEN 'rain'
            WHEN weather_condition_code IN (12, 13, 14, 15, 16, 19, 20, 21, 22) THEN 'snow_or_sleet'
            WHEN weather_condition_code BETWEEN 23 AND 27 THEN 'storm'
            ELSE 'unknown'
        END AS weather_condition
    FROM integrated_taxi_trips
    WHERE weather_match_status = 'exact_hour'
      AND trip_distance IS NOT NULL
)
SELECT
    weather_condition,
    COUNT(trip_id) AS trip_count,
    ROUND(AVG(trip_distance), 3) AS average_trip_distance,
    ROUND(STDDEV_POP(trip_distance), 3) AS trip_distance_stddev
FROM classified
GROUP BY weather_condition
ORDER BY average_trip_distance DESC
"""

## Run 1 (measured first): AQE enabled

This is the reversed position. AQE enabled now goes first, so it loses whatever advantage came from running second in the original notebook.

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

first_run_timing = timed_run(spark, "distance_by_weather_aqe_on_FIRST", QUERY_SQL, repetitions=5)
print("Run 1 — AQE enabled, measured first:", first_run_timing)

Run 1 — AQE enabled, measured first: {'query_name': 'distance_by_weather_aqe_on_FIRST', 'row_count': 5, 'repetitions': 5, 'all_times_seconds': [0.8807, 0.4489, 0.3389, 0.3027, 0.4004], 'median_seconds': 0.4004, 'min_seconds': 0.3027, 'max_seconds': 0.8807}


## Run 2 (measured second): AQE disabled

This is the reversed position. AQE disabled now goes second, so it gains whatever advantage the original notebook's AQE-enabled run had.

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

second_run_timing = timed_run(spark, "distance_by_weather_aqe_off_SECOND", QUERY_SQL, repetitions=5)
print("Run 2 — AQE disabled, measured second:", second_run_timing)

Run 2 — AQE disabled, measured second: {'query_name': 'distance_by_weather_aqe_off_SECOND', 'row_count': 5, 'repetitions': 5, 'all_times_seconds': [0.2525, 0.2756, 0.2721, 0.3107, 0.3024], 'median_seconds': 0.2756, 'min_seconds': 0.2525, 'max_seconds': 0.3107}


## Sanity check: physical plan still shows no structural AQE effect

In [5]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
plan_check = capture_explain(spark.sql(QUERY_SQL), mode="formatted")
has_shuffle_read = "AQEShuffleRead" in plan_check
has_final_initial_split = "== Final Plan ==" in plan_check and "== Initial Plan ==" in plan_check
plans_differ = False
if has_final_initial_split:
    final_section = plan_check.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    initial_section = plan_check.split("== Initial Plan ==")[1]
    plans_differ = final_section.strip() != initial_section.strip()

print("AQEShuffleRead present:", has_shuffle_read)
print("Final Plan differs from Initial Plan:", plans_differ)

AQEShuffleRead present: False
Final Plan differs from Initial Plan: False


## Interpretation

Result: whichever configuration was measured second won, in both notebooks.

| | Measured first | Measured second | Which config "won" |
|---|---|---|---|
| Original notebook | AQE disabled: 0.4041s | AQE enabled: 0.2949s | AQE enabled (1.37x) |
| This notebook | AQE enabled: 0.4004s | AQE disabled: 0.2756 | AQE disabled (1.45x) |

The winner flipped along with the position in the sequence, not with the AQE setting itself. This confirms the original 1.37x was caused by measurement order, not by a real AQE effect. The plan check above backs this up too: still no AQEShuffleRead, still no difference between the Final and Initial Plan. AQE shows no real effect on this query.

In [6]:
spark.stop()